In [2]:
from pyspark.sql.functions import (
    col, sha2, trim, upper, coalesce, lit, to_timestamp, to_date,
    current_timestamp
)
from pyspark.sql.types import DoubleType, IntegerType

RAW_PATH = "Files/raw"

StatementMeta(, c019ae48-0dfa-4835-a873-7a11306b38a7, 4, Finished, Available, Finished, False)

In [5]:
df_bronze_users = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/users")

df_silver_users = (
    df_bronze_users
    .drop("_c0", "password")
    .dropDuplicates(["user_id"])
    .filter(col("user_id").isNotNull())
    .select(
        col("user_id"),
        sha2(trim(col("email")), 256).alias("hashed_email"),
        trim(col("name")).alias("user_name"),
        col("Age").cast(IntegerType()).alias("age"),
        upper(trim(col("Gender"))).alias("gender"),
        trim(col("Marital Status")).alias("marital_status"),
        trim(col("Occupation")).alias("occupation"),
        trim(col("Monthly Income")).alias("monthly_income"),
        trim(col("Educational Qualifications")).alias("educational_qualifications"),
        col("Family size").cast(IntegerType()).alias("family_size"),
        current_timestamp().alias("processed_timestamp")
    )
)

# Overwrite the silver_users Delta table with all columns included
df_silver_users.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_users")


StatementMeta(, c019ae48-0dfa-4835-a873-7a11306b38a7, 7, Finished, Available, Finished, True)

In [12]:
# 2. FOOD: Drop '_c0', Harmonize 'f_id' -> 'food_id'
# ==============================================================================
df_bronze_food = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/food")

df_silver_food = (
    df_bronze_food
    .drop("_c0")
    .withColumnRenamed("f_id", "food_id")
    .dropDuplicates(["food_id"])
    .filter(col("food_id").isNotNull())
    .withColumn("item_name", trim(col("item")))
    .withColumn("veg_or_non_veg", upper(trim(col("veg_or_non_veg"))))
    .withColumn("processed_timestamp", current_timestamp())
    .drop("item")
)

df_silver_food.write.format("delta").mode("overwrite").saveAsTable("silver_food")

StatementMeta(, 31487cc7-2fa5-474c-973f-c09f11fc0dae, 14, Finished, Available, Finished, False)

In [13]:
# 3. RESTAURANT: Drop '_c0', Harmonize 'id' -> 'restaurant_id', Clean Metadata
# ==============================================================================
df_bronze_restaurant = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/restaurant")

df_silver_restaurant = (
    df_bronze_restaurant
    .drop("_c0", "link", "menu")  # Drop index artifact and unused links/blobs
    .withColumnRenamed("id", "restaurant_id")
    .dropDuplicates(["restaurant_id"])
    .filter(col("restaurant_id").isNotNull())
    .withColumn("restaurant_name", trim(col("name")))
    .withColumn("city", upper(trim(col("city"))))
    .withColumn("rating", col("rating").cast(DoubleType()))
    .withColumn("rating_count", col("rating_count").cast(IntegerType()))
    .withColumn("cost", col("cost").cast(DoubleType()))
    .withColumn("cuisine", upper(trim(col("cuisine"))))
    .withColumn("lic_no", trim(col("lic_no")))
    .withColumn("address", trim(col("address")))
    .withColumn("processed_timestamp", current_timestamp())
    .drop("name")
)

df_silver_restaurant.write.format("delta").mode("overwrite").saveAsTable("silver_restaurant")

StatementMeta(, 31487cc7-2fa5-474c-973f-c09f11fc0dae, 15, Finished, Available, Finished, False)

In [14]:
# 4. MENU: Drop '_c0', Harmonize 'r_id' -> 'restaurant_id', 'f_id' -> 'food_id'
# ==============================================================================
df_bronze_menu = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/menu")

df_silver_menu = (
    df_bronze_menu
    .drop("_c0")
    .withColumnRenamed("r_id", "restaurant_id")
    .withColumnRenamed("f_id", "food_id")
    .dropDuplicates(["menu_id"])
    .filter(col("menu_id").isNotNull())
    .withColumn("cuisine", upper(trim(col("cuisine"))))
    .withColumn("price", col("price").cast(DoubleType()))
    .withColumn("processed_timestamp", current_timestamp())
)

df_silver_menu.write.format("delta").mode("overwrite").saveAsTable("silver_menu")

StatementMeta(, 31487cc7-2fa5-474c-973f-c09f11fc0dae, 16, Finished, Available, Finished, False)

In [15]:
# 5. ORDER ITEMS: Harmonize 'r_id' -> 'restaurant_id', 'f_id' -> 'food_id'
# ==============================================================================
df_bronze_order_items = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/order_items")

df_silver_order_items = (
    df_bronze_order_items
    .withColumnRenamed("r_id", "restaurant_id")
    .withColumnRenamed("f_id", "food_id")
    .dropDuplicates(["order_item_id"])
    .filter(col("order_item_id").isNotNull())
    .withColumn("price", col("price").cast(DoubleType()))
    .withColumn("quantity", col("quantity").cast(IntegerType()))
    .withColumn("line_amount", col("line_amount").cast(DoubleType()))
    .withColumn("processed_timestamp", current_timestamp())
)

df_silver_order_items.write.format("delta").mode("overwrite").saveAsTable("silver_order_items")

StatementMeta(, 31487cc7-2fa5-474c-973f-c09f11fc0dae, 17, Finished, Available, Finished, False)

In [16]:
# 6. ORDERS: Harmonize 'r_id' -> 'restaurant_id', Cast Financials & Timestamps
# ==============================================================================
df_bronze_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/orders")

df_silver_orders = (
    df_bronze_orders
    .withColumnRenamed("r_id", "restaurant_id")
    .dropDuplicates(["order_id"])
    .filter(col("order_id").isNotNull())
    .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("restaurant_city", upper(trim(col("restaurant_city"))))
    .withColumn("cuisine", upper(trim(col("cuisine"))))
    .withColumn("items_count", col("items_count").cast(IntegerType()))
    .withColumn("sales_qty", col("sales_qty").cast(IntegerType()))
    .withColumn("subtotal", col("subtotal").cast(DoubleType()))
    .withColumn("discount", col("discount").cast(DoubleType()))
    .withColumn("delivery_fee", col("delivery_fee").cast(DoubleType()))
    .withColumn("gst", col("gst").cast(DoubleType()))
    .withColumn("sales_amount", col("sales_amount").cast(DoubleType()))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("order_status", upper(trim(col("order_status"))))
    .withColumn("customer_rating", col("customer_rating").cast(DoubleType()))
    .withColumn("delivery_time_min", col("delivery_time_min").cast(IntegerType()))
    .withColumn("processed_timestamp", current_timestamp())
)

df_silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 31487cc7-2fa5-474c-973f-c09f11fc0dae, 18, Finished, Available, Finished, False)

In [17]:
# 7. REVIEWS: Standardize Ratings, Dates, Text
# ==============================================================================
df_bronze_reviews = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{RAW_PATH}/reviews")

df_silver_reviews = (
    df_bronze_reviews
    .dropDuplicates(["review_id"])
    .filter(col("review_id").isNotNull())
    .withColumn("rating", col("rating").cast(DoubleType()))
    .withColumn("comment", trim(col("comment")))
    .withColumn("review_date", to_date(col("review_date")))
    .withColumn("processed_timestamp", current_timestamp())
)

df_silver_reviews.write.format("delta").mode("overwrite").saveAsTable("silver_reviews")

StatementMeta(, 31487cc7-2fa5-474c-973f-c09f11fc0dae, 19, Finished, Available, Finished, True)